# 🧠 K-Nearest Neighbors (KNN) Explanation and Hands-on Example

Welcome to the hands-on explanation notebook for **K-Nearest Neighbors (KNN)**! In this notebook, we will:
1. Generate a synthetic 2D dataset representing bounding box feature embeddings for `lever-handle` and `handwheel-handle`.
2. Classify points using standard `scikit-learn`'s `KNeighborsClassifier`.
3. Visualize the decision boundaries for different values of $K$ ($K=1$, $K=5$, $K=20$) to understand the bias-variance tradeoff.
4. Implement **KNN from scratch** in pure Python/NumPy using Euclidean distance and majority voting.
5. Evaluate our scratch implementation on test points and verify its predictions against scikit-learn.
6. Discuss how KNN is used in modern Computer Vision workflows (like Re-identification and Vector Databases).

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from collections import Counter

# Set seed for reproducibility
np.random.seed(42)

## 1. Case Study Data Generation

We generate 80 samples of 2D bounding box visual embedding features:
*   Class 1 (`lever-handle`): centered around (2.0, 3.0)
*   Class 0 (`handwheel-handle`): centered around (4.5, 4.0)

In [ ]:
m = 80

# Class 1: Lever Handles
X_lever = np.random.randn(m // 2, 2) * 0.8 + np.array([2.0, 3.0])
y_lever = np.ones(m // 2)

# Class 0: Handwheel Handles
X_handwheel = np.random.randn(m // 2, 2) * 0.8 + np.array([4.5, 4.0])
y_handwheel = np.zeros(m // 2)

# Combine datasets
X_train = np.vstack((X_lever, X_handwheel))
y_train = np.concatenate((y_lever, y_handwheel))

# Plot the dataset
plt.figure(figsize=(8, 5))
plt.scatter(X_lever[:, 0], X_lever[:, 1], color='blue', label='Class 1: Lever Handle', alpha=0.7)
plt.scatter(X_handwheel[:, 0], X_handwheel[:, 1], color='red', label='Class 0: Handwheel Handle', alpha=0.7)
plt.xlabel('Embedding Feature 1')
plt.ylabel('Embedding Feature 2')
plt.title('Bounding Box Visual Embedding Space')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 2. Decision Boundary Analysis (Bias-Variance Tradeoff)

The choice of $K$ is critical. Let's fit scikit-learn models for $K=1$, $K=5$, and $K=20$ and plot their decision boundaries.

In [ ]:
# Create grid for boundary plotting
x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05),
                     np.arange(y_min, y_max, 0.05))
grid_points = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
cmap_bold = ['#FF0000', '#0000FF']

K_values = [1, 5, 20]

for idx, K in enumerate(K_values):
    clf = KNeighborsClassifier(n_neighbors=K)
    clf.fit(X_train, y_train)
    
    Z = clf.predict(grid_points)
    Z = Z.reshape(xx.shape)
    
    ax = axes[idx]
    ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.6)
    ax.scatter(X_train[:, 0], X_train[:, 1], c=[cmap_bold[int(i)] for i in y_train], edgecolor='k', s=40)
    ax.set_title(f'KNN Boundary (K={K})')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

*   **$K=1$ (Overfitting):** The decision boundary is highly detailed and jagged, fitting around every noise point.
*   **$K=5$ (Ideal Fit):** The boundary is smooth, capturing the general division while ignoring minor noise.
*   **$K=20$ (Underfitting):** The boundary is too simple, missing finer classification patterns.

## 3. KNN from Scratch using NumPy

Let's implement the KNN classifier. The model memorizes training points, then for each query point:
1.  Calculates the Euclidean distance to all training samples.
2.  Sorts the distances.
3.  Takes the labels of the top $K$ nearest points.
4.  Performs a majority vote.

In [ ]:
class CustomKNN:
    def __init__(self, K=5):
        self.K = K
        self.X_train = None
        self.y_train = None
        
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        
    def _predict_single(self, x_query):
        # 1. Compute Euclidean distance
        distances = np.sqrt(np.sum((self.X_train - x_query) ** 2, axis=1))
        
        # 2. Get the indices of the K smallest distances
        k_indices = np.argsort(distances)[:self.K]
        
        # 3. Get the labels of the K nearest neighbors
        k_nearest_labels = self.y_train[k_indices].astype(int)
        
        # 4. Perform a majority vote
        most_common = Counter(k_nearest_labels).most_common(1)
        return most_common[0][0]
        
    def predict(self, X_queries):
        return np.array([self._predict_single(x) for x in X_queries])

# Let's test our scratch implementation
scratch_knn = CustomKNN(K=5)
scratch_knn.fit(X_train, y_train)

# Fit scikit-learn model for comparison
sklearn_knn = KNeighborsClassifier(n_neighbors=5)
sklearn_knn.fit(X_train, y_train)

# Predict on some test points
test_points = np.array([
    [1.5, 2.5],  # Deep inside Class 1
    [4.0, 4.5],  # Deep inside Class 0
    [3.1, 3.5]   # Border area
])

preds_scratch = scratch_knn.predict(test_points)
preds_sklearn = sklearn_knn.predict(test_points)

print("Predictions comparison:")
for i, pt in enumerate(test_points):
    print(f"Point {pt} | Scratch Pred: {preds_scratch[i]} | Sklearn Pred: {preds_sklearn[i]}")

## 4. Performance Validation

Let's check the accuracy of our Custom KNN model on the entire training set.

In [ ]:
y_pred_scratch = scratch_knn.predict(X_train)
y_pred_sklearn = sklearn_knn.predict(X_train)

accuracy_scratch = accuracy_score(y_train, y_pred_scratch)
accuracy_sklearn = accuracy_score(y_train, y_pred_sklearn)

print(f"Custom Scratch KNN Accuracy: {accuracy_scratch * 100:.2f}%")
print(f"Scikit-Learn KNN Accuracy  : {accuracy_sklearn * 100:.2f}%")

## 💡 Connection to Deep Learning & YOLO
*   **Object Re-Identification (Re-ID):** In multi-object tracking (like tracking people or cars across cameras), deep networks extract feature embeddings for each bounding box. Rather than running a classification layer, trackers compute the cosine or Euclidean distance between the new detection embedding and past tracked embeddings to identify the object. This is essentially a 1-Nearest Neighbor search!
*   **Vector Search & Databases:** Modern AI retrieval systems (Retrieval-Augmented Generation / image search) utilize Vector Databases (like Milvus, Pinecone, or FAISS) to run highly optimized KNN queries over millions of high-dimensional vectors.